In [ ]:
import requests
import os
import sys
import platform
from lakehouse import bronze
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

In [2]:
if platform.system() == "Windows":
    os.environ["PYSPARK_PYTHON"] = sys.executable
    os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
    print("Adding Python ENV variables on Windows")

Adding Python ENV variables on Windows


In [3]:
builder = (
    SparkSession.builder.appName("Data with Nikk the Greek Spark Session")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [4]:
CATALOG = spark.catalog.currentCatalog()

# 1. Set Up

In [5]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")

DataFrame[]

In [6]:
options = {
    "catalog": CATALOG,
    "target_schema": "bronze",
}

# 2 Overwrite

In [7]:
class StarWarsBronze(bronze.Bronze):
    def custom_load(self, table):
        results = []
        query = f"https://swapi.tech/api/{table}"
        json_request = requests.get(query).json()
        results.extend(json_request["results"])

        while json_request["next"]:
            json_request = requests.get(json_request["next"]).json()
            results.extend(json_request["results"])
        return spark.createDataFrame(results)


bronze_instance = StarWarsBronze(spark, **options)

In [8]:
(
    bronze_instance.load()
    .transform()
    .write(mode="overwrite")
    .tblproperties(clusterby=["LH_BronzeTS"])
    .optimize(optimize=True, vacuum=True, analyze=False, excl_cols=["url"])
    .execute("people", "planets")
)
spark.sql(f"SELECT * FROM {CATALOG}.bronze.people").show(truncate=False)

2025-02-27 16:08:58 | people | execute | Started
2025-02-27 16:08:58 | people | load | Started
2025-02-27 16:09:04 | people | load | Completed in 0.1 min
2025-02-27 16:09:04 | people | transform | Started
2025-02-27 16:09:04 | people | transform | Completed in 0.0 min
2025-02-27 16:09:04 | people | write | Started
2025-02-27 16:09:36 | people | write | Completed in 0.52 min
2025-02-27 16:09:36 | people | tblproperties | Started
2025-02-27 16:10:15 | people | tblproperties | Completed in 0.63 min
2025-02-27 16:10:15 | people | optimize | Started
2025-02-27 16:11:17 | people | optimize | Completed in 1.02 min
2025-02-27 16:11:17 | people | execute | Completed in 2.3 min
2025-02-27 16:11:17 | planets | execute | Started
2025-02-27 16:11:17 | planets | load | Started
2025-02-27 16:11:20 | planets | load | Completed in 0.05 min
2025-02-27 16:11:20 | planets | transform | Started
2025-02-27 16:11:20 | planets | transform | Completed in 0.0 min
2025-02-27 16:11:20 | planets | write | Started


+--------------------------+-------------------+---+------------------------------------+
|LH_BronzeTS               |name               |uid|url                                 |
+--------------------------+-------------------+---+------------------------------------+
|2025-02-27 16:09:07.015034|Cliegg Lars        |62 |https://www.swapi.tech/api/people/62|
|2025-02-27 16:09:07.015034|Poggle the Lesser  |63 |https://www.swapi.tech/api/people/63|
|2025-02-27 16:09:07.015034|Luminara Unduli    |64 |https://www.swapi.tech/api/people/64|
|2025-02-27 16:09:07.015034|Barriss Offee      |65 |https://www.swapi.tech/api/people/65|
|2025-02-27 16:09:07.015034|Dormé              |66 |https://www.swapi.tech/api/people/66|
|2025-02-27 16:09:07.015034|Dooku              |67 |https://www.swapi.tech/api/people/67|
|2025-02-27 16:09:07.015034|Bail Prestor Organa|68 |https://www.swapi.tech/api/people/68|
|2025-02-27 16:09:07.015034|Jango Fett         |69 |https://www.swapi.tech/api/people/69|
|2025-02-2

In [9]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.people")
print(f"No. Rows: {df.count()}")
df.show(100)

No. Rows: 82
+--------------------+--------------------+---+--------------------+
|         LH_BronzeTS|                name|uid|                 url|
+--------------------+--------------------+---+--------------------+
|2025-02-27 16:09:...|         Cliegg Lars| 62|https://www.swapi...|
|2025-02-27 16:09:...|   Poggle the Lesser| 63|https://www.swapi...|
|2025-02-27 16:09:...|     Luminara Unduli| 64|https://www.swapi...|
|2025-02-27 16:09:...|       Barriss Offee| 65|https://www.swapi...|
|2025-02-27 16:09:...|               Dormé| 66|https://www.swapi...|
|2025-02-27 16:09:...|               Dooku| 67|https://www.swapi...|
|2025-02-27 16:09:...| Bail Prestor Organa| 68|https://www.swapi...|
|2025-02-27 16:09:...|          Jango Fett| 69|https://www.swapi...|
|2025-02-27 16:09:...|          Zam Wesell| 70|https://www.swapi...|
|2025-02-27 16:09:...|     Dexter Jettster| 71|https://www.swapi...|
|2025-02-27 16:09:...|             Lama Su| 72|https://www.swapi...|
|2025-02-27 16:09:...

In [10]:
df = spark.sql(f"SELECT * FROM {CATALOG}.bronze.planets")
print(f"No. Rows: {df.count()}")
df.show(100)

No. Rows: 60
+--------------------+--------------+---+--------------------+
|         LH_BronzeTS|          name|uid|                 url|
+--------------------+--------------+---+--------------------+
|2025-02-27 16:11:...|       Mygeeto| 16|https://www.swapi...|
|2025-02-27 16:11:...|       Felucia| 17|https://www.swapi...|
|2025-02-27 16:11:...|Cato Neimoidia| 18|https://www.swapi...|
|2025-02-27 16:11:...|     Saleucami| 19|https://www.swapi...|
|2025-02-27 16:11:...|       Stewjon| 20|https://www.swapi...|
|2025-02-27 16:11:...|        Eriadu| 21|https://www.swapi...|
|2025-02-27 16:11:...|      Corellia| 22|https://www.swapi...|
|2025-02-27 16:11:...|         Rodia| 23|https://www.swapi...|
|2025-02-27 16:11:...|     Nal Hutta| 24|https://www.swapi...|
|2025-02-27 16:11:...|     Dantooine| 25|https://www.swapi...|
|2025-02-27 16:11:...|    Bestine IV| 26|https://www.swapi...|
|2025-02-27 16:11:...|   Ord Mantell| 27|https://www.swapi...|
|2025-02-27 16:11:...|       unknown| 28|h

In [11]:
bronze_instance.data["people"].show()

+--------------------+--------------------+---+--------------------+
|         LH_BronzeTS|                name|uid|                 url|
+--------------------+--------------------+---+--------------------+
|2025-02-27 16:12:...|      Luke Skywalker|  1|https://www.swapi...|
|2025-02-27 16:12:...|               C-3PO|  2|https://www.swapi...|
|2025-02-27 16:12:...|               R2-D2|  3|https://www.swapi...|
|2025-02-27 16:12:...|         Darth Vader|  4|https://www.swapi...|
|2025-02-27 16:12:...|         Leia Organa|  5|https://www.swapi...|
|2025-02-27 16:12:...|           Owen Lars|  6|https://www.swapi...|
|2025-02-27 16:12:...|  Beru Whitesun lars|  7|https://www.swapi...|
|2025-02-27 16:12:...|               R5-D4|  8|https://www.swapi...|
|2025-02-27 16:12:...|   Biggs Darklighter|  9|https://www.swapi...|
|2025-02-27 16:12:...|      Obi-Wan Kenobi| 10|https://www.swapi...|
|2025-02-27 16:12:...|    Anakin Skywalker| 11|https://www.swapi...|
|2025-02-27 16:12:...|      Wilhuf

In [12]:
bronze_instance.data["planets"].show()

+--------------------+--------------+---+--------------------+
|         LH_BronzeTS|          name|uid|                 url|
+--------------------+--------------+---+--------------------+
|2025-02-27 16:12:...|      Tatooine|  1|https://www.swapi...|
|2025-02-27 16:12:...|      Alderaan|  2|https://www.swapi...|
|2025-02-27 16:12:...|      Yavin IV|  3|https://www.swapi...|
|2025-02-27 16:12:...|          Hoth|  4|https://www.swapi...|
|2025-02-27 16:12:...|       Dagobah|  5|https://www.swapi...|
|2025-02-27 16:12:...|        Bespin|  6|https://www.swapi...|
|2025-02-27 16:12:...|         Endor|  7|https://www.swapi...|
|2025-02-27 16:12:...|         Naboo|  8|https://www.swapi...|
|2025-02-27 16:12:...|     Coruscant|  9|https://www.swapi...|
|2025-02-27 16:12:...|        Kamino| 10|https://www.swapi...|
|2025-02-27 16:12:...|      Geonosis| 11|https://www.swapi...|
|2025-02-27 16:12:...|        Utapau| 12|https://www.swapi...|
|2025-02-27 16:12:...|      Mustafar| 13|https://www.sw

# 6 Clean Up

In [13]:
spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.bronze CASCADE")

DataFrame[]